# SmolLM2-135M: SFT → DPO → GRPO on GSM8K

This GPU notebook demonstrates Picotron's three script-first adaptation stages. GRPO uses a transparent binary reward: a completion earns `1.0` only when its extracted final number matches GSM8K's `#### answer`.


## 1. Clone and install

Enable a Kaggle GPU before running. This notebook starts from the public base model so all three stages are reproducible in one run. If a previous SFT+DPO artifact is attached, the optional reuse cell below can skip those stages.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path('/kaggle/working/picotron')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/AndyMLAndAI/picotron.git', str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'accelerate'], check=True)


## 2. Load the real Hugging Face model

SmolLM2 is loaded through Transformers, demonstrating that Picotron's SFT/DPO/GRPO layers operate on ordinary HF causal LMs rather than the native Picotron decoder alone.


In [ ]:
import itertools
import re
from dataclasses import replace

import torch
from datasets import load_dataset
from torch.utils.data import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from picotron.config.config import load_config
from picotron_sft import run_sft
from picotron_dpo import run_dpo
from picotron_grpo import run_grpo

MODEL_ID = 'HuggingFaceTB/SmolLM2-135M'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'Enable Kaggle GPU acceleration.'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).to(DEVICE)
model.config.use_cache = False

hf_config = model.config
print({name: getattr(hf_config, name, 'N/A') for name in ('model_type', 'hidden_size', 'num_hidden_layers', 'num_attention_heads', 'num_key_value_heads', 'vocab_size', 'rope_theta')})

base_display_config = load_config(REPO / 'src/picotron/config/picotron_decoder.yaml')
display_config = replace(
    base_display_config,
    model=replace(base_display_config.model, model_config=replace(
        base_display_config.model.model_config, vocab_size=hf_config.vocab_size,
        hidden_size=hf_config.hidden_size, intermediate_size=hf_config.intermediate_size,
        num_hidden_layers=hf_config.num_hidden_layers, num_attention_heads=hf_config.num_attention_heads,
        num_key_value_heads=getattr(hf_config, 'num_key_value_heads', None),
    )),
    data=replace(base_display_config.data, vocab_size=hf_config.vocab_size),
    tokens=replace(base_display_config.tokens, sequence_length=256, micro_batch_size=2),
    optimizer=replace(base_display_config.optimizer, learning_rate_scheduler=replace(base_display_config.optimizer.learning_rate_scheduler, learning_rate=2e-5)),
)


## 3. Baseline GSM8K-style inference

These prompts are retained for the final comparison. The held-out GSM8K slice below provides the quantitative metric.


In [ ]:
MATH_PROMPTS = [
    'If a box has 7 red balls and 5 blue balls, how many balls are in the box? Give only the final number.',
    'A baker makes 12 cookies each day for 4 days. How many cookies are made? Give only the final number.',
    'What is 15 multiplied by 6? Give only the final number.',
]
GENERAL_PROMPTS = ['Explain why the sky appears blue in two sentences.', 'Write a short Python function that checks whether a number is prime.']

def generate(active_model, prompt, max_new_tokens=96, do_sample=False):
    active_model.eval()
    previous = active_model.config.use_cache
    active_model.config.use_cache = True
    try:
        encoded = tokenizer(prompt, return_tensors='pt').to(DEVICE)
        generated = active_model.generate(**encoded, max_new_tokens=max_new_tokens, do_sample=do_sample, temperature=0.8 if do_sample else None, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
        return tokenizer.decode(generated[0], skip_special_tokens=True)
    finally:
        active_model.config.use_cache = previous

baseline_math = [generate(model, prompt) for prompt in MATH_PROMPTS]
for prompt, output in zip(MATH_PROMPTS, baseline_math):
    print(f'PROMPT: {prompt}\nBASE: {output}\n')


## 4. SFT then DPO

This bounded demo uses 5,000 SmolTalk examples / 500 SFT steps and 2,000 UltraFeedback preferences / 300 DPO steps. The DPO reference is frozen automatically from the already-SFT policy.


In [ ]:
MAX_LENGTH, SFT_EXAMPLES, SFT_STEPS, DPO_EXAMPLES, DPO_STEPS = 256, 5_000, 500, 2_000, 300
smoltalk = list(itertools.islice(load_dataset('HuggingFaceTB/smoltalk', 'all', split='train', streaming=True), SFT_EXAMPLES))
assert len(smoltalk) == SFT_EXAMPLES

class SmolTalkDataset(Dataset):
    def __init__(self, examples): self.examples = examples
    def __len__(self): return len(self.examples)
    def __getitem__(self, index):
        messages = self.examples[index]['messages']
        ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=False) if getattr(tokenizer, 'chat_template', None) else tokenizer.encode('\n'.join(f"{m['role']}: {m['content']}" for m in messages), add_special_tokens=False)
        ids = ids[:MAX_LENGTH]
        padding = MAX_LENGTH - len(ids)
        return {'input_ids': torch.tensor(ids + [tokenizer.pad_token_id] * padding), 'attention_mask': torch.tensor([1] * len(ids) + [0] * padding), 'labels': torch.tensor(ids + [-100] * padding)}

sft_losses = run_sft(model, SmolTalkDataset(smoltalk), learning_rate=2e-5, batch_size=2, num_steps=SFT_STEPS, device=DEVICE, display_config=display_config)

def last_assistant(messages):
    return next((m['content'] for m in reversed(messages) if m.get('role') == 'assistant' and isinstance(m.get('content'), str)), None)
def dpo_prompt(prompt):
    messages = [{'role': 'user', 'content': prompt}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) if getattr(tokenizer, 'chat_template', None) else f'user: {prompt}\nassistant: '

triples = []
for example in load_dataset('HuggingFaceH4/ultrafeedback_binarized', split='train_prefs', streaming=True):
    chosen, rejected = last_assistant(example['chosen']), last_assistant(example['rejected'])
    if isinstance(example['prompt'], str) and chosen and rejected: triples.append((dpo_prompt(example['prompt']), chosen, rejected))
    if len(triples) == DPO_EXAMPLES: break
assert len(triples) == DPO_EXAMPLES
dpo_losses = run_dpo(model, triples, tokenizer=tokenizer, beta=0.1, learning_rate=1e-5, batch_size=1, max_length=MAX_LENGTH, num_steps=DPO_STEPS, device=DEVICE, display_config=display_config)
DPO_DIR = REPO / 'artifacts' / 'smollm2_135m_sft_dpo'
DPO_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(DPO_DIR); tokenizer.save_pretrained(DPO_DIR)
print(f'SFT {sft_losses[0]:.4f} → {sft_losses[-1]:.4f}; DPO {dpo_losses[0]:.4f} → {dpo_losses[-1]:.4f}')


## 5. GSM8K reward design and extractor sanity check

GSM8K stores the target after `####`. Completions are parsed conservatively: boxed answers, `the answer is`, then the final numeric-looking token. We print examples before training; stop here if targets are not parsed correctly.


In [ ]:
NUMBER = r'[-+]?\d[\d,]*(?:\.\d+)?'

def normalize_number(value):
    if value is None: return None
    value = value.replace(',', '').strip().rstrip('.')
    try:
        numeric = float(value)
        return str(int(numeric)) if numeric.is_integer() else str(numeric)
    except ValueError:
        return None

def extract_final_number(text):
    patterns = [r'\\boxed\{([^}]*)\}', rf'(?:final answer|the answer is|answer is)\s*[:=]?\s*({NUMBER})', rf'({NUMBER})(?!.*{NUMBER})']
    for pattern in patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE | re.DOTALL)
        if matches:
            value = matches[-1]
            if isinstance(value, tuple): value = value[-1]
            normalized = normalize_number(value)
            if normalized is not None: return normalized
    return None

def extract_gsm8k_target(answer):
    match = re.search(r'####\s*(' + NUMBER + r')', answer)
    return normalize_number(match.group(1)) if match else None

GRPO_EXAMPLES, HELDOUT_EXAMPLES = 300, 50
gsm8k = load_dataset('openai/gsm8k', 'main', split='train')
records = []
for example in itertools.islice(gsm8k, GRPO_EXAMPLES + HELDOUT_EXAMPLES):
    target = extract_gsm8k_target(example['answer'])
    assert target is not None, f'Could not parse GSM8K target: {example["answer"][-100:]}'
    prompt = example['question'] + '\nSolve step by step, then state the final answer clearly.'
    records.append({'prompt': prompt, 'target': target, 'answer': example['answer']})
train_records, heldout_records = records[:GRPO_EXAMPLES], records[GRPO_EXAMPLES:]
targets = {item['prompt']: item['target'] for item in train_records}

for item in train_records[:3]:
    sample = generate(model, item['prompt'], max_new_tokens=128, do_sample=False)
    print('TARGET:', item['target'], 'EXTRACTED:', extract_final_number(sample))
    print('COMPLETION:', sample[-300:], '\n')

def gsm8k_reward(prompt, completion):
    return float(extract_final_number(completion) == targets[prompt])


## 6. GRPO on GSM8K

Each prompt produces a group of sampled completions. Rewards are normalized within that group; the frozen DPO policy supplies the KL reference. The live table reports mean group reward, advantage magnitude, KL, and clipped policy loss.


In [ ]:
def heldout_accuracy(records):
    outcomes = []
    for item in records:
        completion = generate(model, item['prompt'], max_new_tokens=128, do_sample=False)
        outcomes.append(extract_final_number(completion) == item['target'])
    return sum(outcomes) / len(outcomes), outcomes

baseline_heldout_accuracy, _ = heldout_accuracy(heldout_records)
print(f'Held-out GSM8K exact-final-number accuracy before GRPO: {baseline_heldout_accuracy:.1%}')


In [ ]:
GRPO_STEPS = 300
grpo_losses = run_grpo(
    model, [item['prompt'] for item in train_records], gsm8k_reward, tokenizer=tokenizer,
    group_size=4, beta=0.04, clip_epsilon=0.2, learning_rate=5e-6,
    max_new_tokens=128, temperature=0.8, num_steps=GRPO_STEPS,
    device=DEVICE, display_config=display_config,
)
GRPO_DIR = REPO / 'artifacts' / 'smollm2_135m_sft_dpo_grpo_gsm8k'
GRPO_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(GRPO_DIR); tokenizer.save_pretrained(GRPO_DIR)
print(f'GRPO complete: first_loss={grpo_losses[0]:.4f}, last_loss={grpo_losses[-1]:.4f}')


## 7. Post-GRPO inference

Compare the same math and general prompts. A small math-only run may not visibly improve every prompt; the held-out metric below is the meaningful signal.


In [ ]:
for prompt, before in zip(MATH_PROMPTS, baseline_math):
    print('=' * 80)
    print('PROMPT:', prompt)
    print('BASE:', before)
    print('POST-GRPO:', generate(model, prompt))

for prompt in GENERAL_PROMPTS:
    print('=' * 80)
    print('GENERAL PROMPT:', prompt)
    print('POST-GRPO:', generate(model, prompt))


## 8. Held-out GSM8K score

These 50 examples were excluded from GRPO. The score is greedy exact-final-number accuracy, so it measures the same reward signal without sampling noise.


In [ ]:
after_accuracy, outcomes = heldout_accuracy(heldout_records)
print(f'Held-out GSM8K exact-final-number accuracy: {baseline_heldout_accuracy:.1%} → {after_accuracy:.1%} ({sum(outcomes)}/{len(outcomes)} after)')
print('Kaggle-only verification: confirm openai/gsm8k main schema, downloads, extractor samples, GPU memory, and GRPO reward trend.')
